## Reflexion Agent (No Framework)

This notebook extends the simple Bedrock + tools agent with a **Reflexion-style loop**:

- attempt an answer
- **verify** it with a Python checker
- if wrong, generate a short **reflection** (what went wrong + how to fix)
- retry using the reflection as additional context

Goal: the agent should reliably solve the puzzle:

> In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?



## Setup

Set these environment variables (recommended) or configure an AWS profile:

- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_REGION` (e.g. `ap-southeast-1`)
- `AWS_BEDROCK_MODEL_ID` (optional)

Then run the cells top-to-bottom.


In [ ]:
# copy the info here from the aws_credential.txt file


In [19]:
import os

AWS_REGION = os.getenv("AWS_REGION", "ap-southeast-1")
MODEL_ID = os.getenv("AWS_BEDROCK_MODEL_ID", "apac.anthropic.claude-sonnet-4-20250514-v1:0")

print("AWS_REGION:", AWS_REGION)
print("MODEL_ID:", MODEL_ID)
print("Has AWS_ACCESS_KEY_ID:", bool(os.getenv("AWS_ACCESS_KEY_ID")))
print("Has AWS_PROFILE:", bool(os.getenv("AWS_PROFILE")))



AWS_REGION: ap-southeast-1
MODEL_ID: apac.anthropic.claude-sonnet-4-20250514-v1:0
Has AWS_ACCESS_KEY_ID: False
Has AWS_PROFILE: False


In [20]:
import boto3
from typing import Any

bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)


def bedrock_chat(system_prompt: str, messages: list[dict[str, Any]], *, temperature: float = 0.0, max_tokens: int = 800) -> str:
    """Send messages to Bedrock (Converse) and return assistant text."""
    resp = bedrock_rt.converse(
        modelId=MODEL_ID,
        system=[{"text": system_prompt}],
        messages=messages,
        inferenceConfig={"temperature": temperature, "maxTokens": max_tokens},
    )

    out = resp.get("output", {}).get("message", {}).get("content", [])
    return "".join(part.get("text", "") for part in out)



## Tools

The agent only has a `calculator` tool.

> We keep the brute-force solver **only for the verifier** (ground truth), so the agent cannot call it directly.


In [21]:
from __future__ import annotations

import ast
import re
from typing import Callable


_ALLOWED_MATH_FUNCS = {
    "abs": abs,
    "round": round,
    "min": min,
    "max": max,
    "sum": sum,
    "pow": pow,
}


def _safe_eval_expr(node: ast.AST):
    if isinstance(node, ast.Expression):
        return _safe_eval_expr(node.body)

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
        v = _safe_eval_expr(node.operand)
        return +v if isinstance(node.op, ast.UAdd) else -v

    if isinstance(node, ast.BinOp) and isinstance(
        node.op, (ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow)
    ):
        a = _safe_eval_expr(node.left)
        b = _safe_eval_expr(node.right)
        return {
            ast.Add: lambda x, y: x + y,
            ast.Sub: lambda x, y: x - y,
            ast.Mult: lambda x, y: x * y,
            ast.Div: lambda x, y: x / y,
            ast.FloorDiv: lambda x, y: x // y,
            ast.Mod: lambda x, y: x % y,
            ast.Pow: lambda x, y: x**y,
        }[type(node.op)](a, b)

    if isinstance(node, ast.Call):
        if node.keywords:
            raise ValueError("keyword args not allowed")
        if not isinstance(node.func, ast.Name):
            raise ValueError("only simple function calls allowed")
        fn = _ALLOWED_MATH_FUNCS.get(node.func.id)
        if fn is None:
            raise ValueError("function not allowed")
        args = [_safe_eval_expr(a) for a in node.args]
        return fn(*args)

    raise ValueError(f"unsupported expression: {type(node).__name__}")


def calculator(expression: str | None = None, **kwargs) -> str:
    """Safely evaluate simple arithmetic expressions.

    Accepts either:
    - calculator("1988 - 1966")
    - calculator(expression="1988 - 1966")
    - calculator(operation="subtract", operand1=1988, operand2=1966)
    """
    if expression is None:
        if "expression" in kwargs and isinstance(kwargs["expression"], str):
            expression = kwargs["expression"]
        elif {"operation", "operand1", "operand2"}.issubset(kwargs.keys()):
            op = str(kwargs["operation"]).lower()
            a = kwargs["operand1"]
            b = kwargs["operand2"]
            op_map = {
                "add": "+",
                "plus": "+",
                "subtract": "-",
                "minus": "-",
                "mul": "*",
                "multiply": "*",
                "times": "*",
                "div": "/",
                "divide": "/",
            }
            if op not in op_map:
                raise ValueError(f"unsupported operation: {op}")
            expression = f"{a} {op_map[op]} {b}"
        else:
            raise ValueError("calculator needs 'expression' or (operation, operand1, operand2)")

    tree = ast.parse(str(expression).strip(), mode="eval")
    return str(_safe_eval_expr(tree))


# Agent tools (NOTE: bruteforce is NOT exposed as a tool)
TOOLS: dict[str, Callable[..., object]] = {
    "calculator": calculator,
}

print("Agent tools:", sorted(TOOLS.keys()))



Agent tools: ['calculator']


## Evaluator/Verifier (LLM)

For simplification, here we use the **same LLM** acts as:

- thinker
- evaluator/verifier (self-judge)
- reflectioner (self-critique)

The verifier will return a JSON verdict: `{ "ok": true/false, "feedback": "..." }`.


In [22]:
import json

SELF_VERIFIER_SYSTEM = """You are a strict verifier.

Given a user question and a proposed answer, decide if the answer is correct.
Return ONLY a single JSON object with keys:
- ok: true/false
- feedback: short explanation

Rules:
- No prose outside JSON.
- No markdown.
- Output must be valid JSON.

Example:
{"ok": true, "feedback": "Correct."}
"""


def _extract_json_object(text: str) -> str | None:
    """Return the first JSON object substring, or None if not found."""
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start : i + 1]
    return None


def self_verify_llm(user_question: str, answer_text: str) -> dict:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "text": f"Question:\n{user_question}\n\nProposed answer:\n{answer_text}\n\nReturn JSON verdict only.",
                }
            ],
        }
    ]

    raw = bedrock_chat(SELF_VERIFIER_SYSTEM, messages, temperature=0.0, max_tokens=200).strip()
    json_str = _extract_json_object(raw)
    if json_str is None:
        return {"ok": False, "feedback": f"Verifier returned no JSON: {raw}"}

    try:
        verdict = json.loads(json_str)
    except Exception:
        verdict = {"ok": False, "feedback": f"Verifier returned invalid JSON: {json_str}"}

    if "ok" not in verdict:
        verdict = {"ok": False, "feedback": f"Verifier JSON missing 'ok': {verdict}"}

    if "feedback" not in verdict:
        verdict["feedback"] = "No feedback provided."

    return verdict




## Agent loop (inner)



In [23]:
import json

ACTION_RE = re.compile(r"^Action:\s*(?P<name>\w+)\s*$", re.MULTILINE)
FINAL_RE = re.compile(r"^Final:\s*(?P<final>[\s\S]+)$", re.MULTILINE)


def extract_first_json_object(s: str) -> str:
    start = s.find("{")
    if start == -1:
        raise ValueError("no JSON object found")

    i = start
    depth = 0
    in_str = False
    esc = False

    while i < len(s):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start : i + 1]
        i += 1

    raise ValueError("unterminated JSON")


def _extract_action_input_value(rest: str):
    """Parse Action Input that may be a JSON object OR a JSON string (or raw text).

    Examples the model may output:
    - Action Input: {"expression": "1988 - 1966"}
    - Action Input: "1988 - 1966"
    - Action Input: {"operation": "subtract", "operand1": 1988, "operand2": 1966}
    """
    s = rest.strip()
    if not s:
        raise ValueError("empty Action Input")

    # If it starts with a JSON object, extract balanced {...}
    if s.startswith("{"):
        return json.loads(extract_first_json_object(s))

    # If it starts with a JSON string, parse the first line as JSON
    if s.startswith('"'):
        first_line = s.splitlines()[0]
        return json.loads(first_line)

    # If it looks like a bare expression (not JSON), treat it as string
    return s.splitlines()[0]


def parse_tool_call(model_text: str) -> tuple[str | None, dict | None, str | None]:
    """Return (tool_name, tool_kwargs, final_text). Prioritize tool call over Final."""
    m_action = ACTION_RE.search(model_text)
    if m_action:
        tool = m_action.group("name")
        after_action = model_text[m_action.end() :]
        m_ai = re.search(r"^Action Input:\s*(?P<rest>[\s\S]+)$", after_action, re.MULTILINE)
        if not m_ai:
            raise ValueError("Action Input not found")

        value = _extract_action_input_value(m_ai.group("rest"))
        if isinstance(value, dict):
            kwargs = value
        elif isinstance(value, str):
            kwargs = {"expression": value}
        else:
            kwargs = {"expression": str(value)}

        return tool, kwargs, None

    m_final = FINAL_RE.search(model_text)
    if m_final:
        return None, None, m_final.group("final").strip()

    return None, None, None


def run_inner_agent(user_prompt: str, *, max_steps: int = 10, temperature: float = 0.0, verbose: bool = True) -> str:
    system_prompt = f"""You are a careful math assistant.

You have access to EXACTLY ONE tool: `calculator`.
Do NOT mention or request any other tools.

TOOLS:
- calculator: arithmetic


Output format:
- Tool call:
Thought: <short>
Action: <tool_name>
Action Input: <valid JSON object>

- Final:
Final: <answer>

Rules:
- Use at most one tool call per message.
- Never include Final in the same message as Action.
"""

    messages: list[dict[str, Any]] = [{"role": "user", "content": [{"text": user_prompt}]}]

    for step in range(1, max_steps + 1):
        text = bedrock_chat(system_prompt, messages, temperature=temperature)
        if verbose:
            print(f"\n--- inner step {step} model ---\n{text}")

        tool, kwargs, final = parse_tool_call(text)
        if final is not None:
            return final

        if tool is None:
            return text.strip()

        if tool not in TOOLS:
            obs = f"ERROR: unknown tool {tool}"
        else:
            try:
                obs = str(TOOLS[tool](**kwargs))
            except Exception as e:
                obs = f"ERROR running {tool}: {type(e).__name__}: {e}"

        if verbose:
            print(f"--- tool --- {tool} {kwargs}")
            print(f"--- observation ---\n{obs}\n")

        messages.append({"role": "assistant", "content": [{"text": text}]})
        messages.append({"role": "user", "content": [{"text": f"Observation: {obs}"}]})

    return "ERROR: exceeded max_steps"



## Reflexion loop (outer)

If the first attempt is wrong, we:

- call the verifier
- ask the model to write a short **Reflection**
- retry with that reflection injected


In [24]:
REFLECTION_SYSTEM = """You are an expert at debugging your own reasoning.

Given:
- the user question
- your previous (wrong) answer
- verifier feedback

Write a short Reflection with:
1) what went wrong
2) what concrete strategy you will use next

Keep it to 3-6 bullet points.
"""


def generate_reflection(user_question: str, previous_answer: str, verifier_feedback: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "text": f"User question:\n{user_question}\n\nPrevious answer:\n{previous_answer}\n\nVerifier feedback:\n{verifier_feedback}",
                }
            ],
        }
    ]
    return bedrock_chat(REFLECTION_SYSTEM, messages, temperature=0.2, max_tokens=300).strip()


def run_reflexion_agent(
    user_question: str,
    *,
    attempts: int = 3,
    inner_max_steps: int = 10,
    verbose: bool = True,
) -> dict:
    reflections: list[str] = []

    for k in range(1, attempts + 1):
        if verbose:
            print(f"\n====================\nAttempt {k}/{attempts}\n====================")

        reflection_text = "\n".join([f"- {r}" for r in reflections])
        injected = (
            "\n\nREFLECTION NOTES (use these to improve your next attempt):\n" + reflection_text
            if reflections
            else ""
        )

        answer = run_inner_agent(
            user_question + injected,
            max_steps=inner_max_steps,
            temperature=0.0,
            verbose=verbose,
        )

        verdict = self_verify_llm(user_question, answer)
        if verbose:
            print("\n--- verifier ---")
            print(verdict)

        if verdict["ok"]:
            return {"ok": True, "answer": answer, "verdict": verdict, "reflections": reflections}

        # generate reflection and retry
        reflection = generate_reflection(user_question, answer, verdict["feedback"])
        reflections.append(reflection)
        if verbose:
            print("\n--- reflection ---")
            print(reflection)

    return {"ok": False, "answer": answer, "verdict": verdict, "reflections": reflections}



## Demo: the 1988 age puzzle


In [25]:
question = "In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?"

result = run_reflexion_agent(question, attempts=3, inner_max_steps=8, verbose=True)

print("\n=== FINAL RESULT ===")
print(result)




Attempt 1/3

--- inner step 1 model ---
I need to find a person whose age in 1988 equals the sum of the digits of their birth year.

Let me set up the equation. If someone was born in year Y, then:
- Their age in 1988 = 1988 - Y
- Sum of digits of Y = their age in 1988

So: 1988 - Y = sum of digits of Y

Let me try birth years around the reasonable range. Since we're looking at 1988, let's try years in the 1900s first.

For a birth year like 19XY, the sum of digits would be 1 + 9 + X + Y = 10 + X + Y.

So: 1988 - (1900 + 10X + Y) = 10 + X + Y
88 - 10X - Y = 10 + X + Y
78 = 11X + 2Y

Let me try X = 7: 78 = 77 + 2Y, so 2Y = 1, Y = 0.5 (not an integer)
Let me try X = 6: 78 = 66 + 2Y, so 2Y = 12, Y = 6

So the birth year would be 1966.

Thought: Let me verify this calculation - if born in 1966, age in 1988 would be 1988 - 1966, and sum of digits of 1966 would be 1 + 9 + 6 + 6.

Action: calculator
Action Input: {"operation": "subtract", "operand1": 1988, "operand2": 1966}
--- tool --- calc

## Demo: Black & red cards

In [26]:
question = "Petya is drawing black and red cards from a bag and stacking them in two piles. It is forbidden to place one card on top of another card of the same color. The tenth and eleventh cards that Petya laid out are red, and the twenty-fifth card is black. What is the color of the twenty-sixth card laid out?"
result = run_reflexion_agent(question, attempts=3, inner_max_steps=8, verbose=True)

print("\n=== FINAL RESULT ===")
print(result)


Attempt 1/3

--- inner step 1 model ---
I need to analyze this step-by-step to understand the pattern of card placement.

Given information:
- Cards must alternate colors in each pile (no same color on top of each other)
- 10th and 11th cards are red
- 25th card is black
- Need to find the color of the 26th card

Let me think about this systematically. Since the 10th and 11th cards are both red, they must be placed on different piles (since you can't place a red card on top of another red card).

Thought: I need to figure out the alternating pattern. If cards alternate colors in each pile, and the 10th and 11th cards are both red, I can work out the pattern from there.

Action: calculator
Action Input: {"operation": "subtract", "operands": [11, 10]}
--- tool --- calculator {'operation': 'subtract', 'operands': [11, 10]}
--- observation ---
ERROR running calculator: ValueError: calculator needs 'expression' or (operation, operand1, operand2)


--- inner step 2 model ---
Thought: Let me